# EDA — Yellow Taxi 2023 (understand the data before the pipeline)

Goal: understand *what the raw data actually is* before loading it into Snowflake. We use **DuckDB locally** here — this is exactly the "explore a Parquet file in seconds" case (the production pipeline runs on Snowflake; see `docs/`).

We work on **one month (Jan 2023)** for speed; conclusions generalise to the full year.

> Setup: `pip install -r eda/requirements.txt`

## Data dictionary (Yellow Taxi) — verified against the Parquet

| Column | Type | Description |
|---|---|---|
| `VendorID` | int | Provider that supplied the record. **1**=Creative Mobile Technologies, **2**=VeriFone Inc. |
| `tpep_pickup_datetime` | timestamp | Meter engaged (trip start) |
| `tpep_dropoff_datetime` | timestamp | Meter disengaged (trip end) |
| `passenger_count` | int* | Passengers, **driver-entered** (so unreliable) |
| `trip_distance` | float | Trip distance in miles, by taximeter |
| `RatecodeID` | int* | Final rate code. **1**=Standard, **2**=JFK, **3**=Newark, **4**=Nassau/Westchester, **5**=Negotiated, **6**=Group ride. (`99`/null = unknown — undocumented) |
| `store_and_fwd_flag` | char | **Y**=trip held in vehicle memory before sending (no live connection), **N**=not |
| `PULocationID` | int | TLC Taxi Zone where the meter was engaged |
| `DOLocationID` | int | TLC Taxi Zone where the meter was disengaged |
| `payment_type` | int | **1**=Credit card, **2**=Cash, **3**=No charge, **4**=Dispute, **5**=Unknown, **6**=Voided. (`0` also appears — undocumented) |
| `fare_amount` | float | Time-and-distance fare by the meter |
| `extra` | float | Misc extras/surcharges (e.g. $0.50 / $1 rush-hour & overnight) |
| `mta_tax` | float | $0.50 MTA tax (auto-triggered by meter) |
| `tip_amount` | float | Tip. **Credit-card tips only** — cash tips are NOT recorded |
| `tolls_amount` | float | Total tolls paid on the trip |
| `improvement_surcharge` | float | $0.30 improvement surcharge |
| `total_amount` | float | Total charged to passenger. **Excludes cash tips** |
| `congestion_surcharge` | float | NY State congestion surcharge |
| `airport_fee` | float | $1.25 for pickups at LaGuardia / JFK |

*`passenger_count` and `RatecodeID` are stored as floats in the Parquet (nullable).

Geography (borough/zone) lives only in `taxi_zone_lookup.csv` — joined via `PULocationID`/`DOLocationID`. There is **no lat/long**.

In [ ]:
import os, duckdb, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', None); pd.set_option('display.width', 140)

BASE = 'https://d37ci6vzurychx.cloudfront.net'
DATA = os.path.join('..', 'data', 'yellow_tripdata_2023-01.parquet')
ZONES = os.path.join('..', 'dbt', 'seeds', 'taxi_zone_lookup.csv')

# Use a local copy if ingestion already downloaded it; otherwise pull Jan 2023.
SRC = DATA if os.path.exists(DATA) else f'{BASE}/trip-data/yellow_tripdata_2023-01.parquet'
ZON = ZONES if os.path.exists(ZONES) else f'{BASE}/misc/taxi_zone_lookup.csv'

con = duckdb.connect()
con.sql('INSTALL httpfs; LOAD httpfs; SET http_retries=8;')
con.sql(f"CREATE VIEW trips AS SELECT * FROM read_parquet('{SRC}')")
con.sql(f"CREATE VIEW zones AS SELECT * FROM read_csv_auto('{ZON}')")
print('Source:', SRC)

## 1. Shape, schema & date range

In [ ]:
n = con.sql('SELECT COUNT(*) FROM trips').fetchone()[0]
print(f'{n:,} rows')
con.sql("""
  SELECT MIN(tpep_pickup_datetime) AS min_pickup,
         MAX(tpep_pickup_datetime) AS max_pickup
  FROM trips
""").df()

In [ ]:
con.sql('DESCRIBE trips').df()

**Note the date range** — a "January 2023" file usually contains a few stray rows from other months/years (sensor lag). We'll quantify that below; the pipeline's year filter removes it.

## 2. Missing values per column

In [ ]:
cols = [r[0] for r in con.sql('DESCRIBE trips').fetchall()]
parts = [f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}" for c in cols]
nulls = con.sql(f"SELECT {', '.join(parts)} FROM trips").df().T
nulls.columns = ['null_count']; nulls['pct'] = (100*nulls['null_count']/n).round(3)
nulls[nulls['null_count'] > 0]

## 3. Numeric distributions
Watch for impossible values: negative fares, zero distance, absurd maxima.

In [ ]:
con.sql("""
  SELECT
    ROUND(AVG(trip_distance),2) AS dist_avg, MIN(trip_distance) dist_min, MAX(trip_distance) dist_max,
    ROUND(AVG(fare_amount),2)   AS fare_avg, MIN(fare_amount) fare_min, MAX(fare_amount) fare_max,
    ROUND(AVG(total_amount),2)  AS total_avg, MIN(total_amount) total_min, MAX(total_amount) total_max,
    MIN(passenger_count) pax_min, MAX(passenger_count) pax_max
  FROM trips
""").df().T.rename(columns={0:'value'})

In [ ]:
# Percentiles give a truer picture than mean/max when outliers exist
con.sql("""
  SELECT
    quantile_cont(trip_distance,[0.5,0.9,0.99]) AS dist_p50_p90_p99,
    quantile_cont(fare_amount,  [0.5,0.9,0.99]) AS fare_p50_p90_p99,
    quantile_cont(total_amount, [0.5,0.9,0.99]) AS total_p50_p90_p99
  FROM trips WHERE fare_amount > 0
""").df().T.rename(columns={0:'p50 / p90 / p99'})

In [ ]:
# Trip-distance histogram (clip the long tail at 20 mi)
df = con.sql('SELECT trip_distance FROM trips WHERE trip_distance BETWEEN 0 AND 20').df()
df.hist(bins=40, figsize=(9,3)); plt.title('Trip distance (0-20 mi)'); plt.tight_layout(); plt.show()

## 4. Categorical columns (with dictionary labels)
Reveals the **undocumented codes** in real data.

In [ ]:
con.sql("""
  SELECT payment_type,
         CASE payment_type WHEN 0 THEN '(0) undocumented' WHEN 1 THEN 'Credit card'
              WHEN 2 THEN 'Cash' WHEN 3 THEN 'No charge' WHEN 4 THEN 'Dispute'
              WHEN 5 THEN 'Unknown' WHEN 6 THEN 'Voided' ELSE '???' END AS label,
         COUNT(*) AS trips,
         ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),2) AS pct
  FROM trips GROUP BY 1,2 ORDER BY trips DESC
""").df()

In [ ]:
con.sql("""
  SELECT RatecodeID,
         CASE RatecodeID WHEN 1 THEN 'Standard' WHEN 2 THEN 'JFK' WHEN 3 THEN 'Newark'
              WHEN 4 THEN 'Nassau/Westchester' WHEN 5 THEN 'Negotiated' WHEN 6 THEN 'Group'
              WHEN 99 THEN '(99) unknown' ELSE 'null' END AS label,
         COUNT(*) AS trips
  FROM trips GROUP BY 1,2 ORDER BY trips DESC
""").df()

In [ ]:
for c in ['VendorID', 'store_and_fwd_flag']:
    print(con.sql(f'SELECT {c}, COUNT(*) AS trips FROM trips GROUP BY 1 ORDER BY trips DESC').df().to_string(index=False)); print()

## 5. Data-quality red flags → why the pipeline filters exist
Each count below motivates a filter in the dbt intermediate layer.

In [ ]:
con.sql(f"""
  SELECT
    SUM(CASE WHEN fare_amount<=0 THEN 1 ELSE 0 END)                        AS fare_le_0,
    SUM(CASE WHEN trip_distance<=0 THEN 1 ELSE 0 END)                      AS dist_le_0,
    SUM(CASE WHEN total_amount<fare_amount THEN 1 ELSE 0 END)              AS total_lt_fare,
    SUM(CASE WHEN tpep_dropoff_datetime<=tpep_pickup_datetime THEN 1 ELSE 0 END) AS dropoff_le_pickup,
    SUM(CASE WHEN passenger_count IS NULL OR passenger_count<1 OR passenger_count>6 THEN 1 ELSE 0 END) AS pax_out_of_range,
    SUM(CASE WHEN datediff('minute',tpep_pickup_datetime,tpep_dropoff_datetime) > 180 THEN 1 ELSE 0 END) AS over_3h,
    SUM(CASE WHEN EXTRACT(YEAR FROM tpep_pickup_datetime)<>2023 THEN 1 ELSE 0 END) AS not_2023
  FROM trips
""").df().T.rename(columns={0:'rows'})

## 6. Geography — join the zone lookup
Top pickup boroughs, and the share of trips with **unknown geography** (IDs 264/265).

In [ ]:
con.sql("""
  SELECT z.Borough AS borough, COUNT(*) AS trips,
         ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (),1) AS pct
  FROM trips t JOIN zones z ON t.PULocationID=z.LocationID
  GROUP BY 1 ORDER BY trips DESC
""").df()

In [ ]:
con.sql("""
  SELECT z.Borough, z.Zone, COUNT(*) AS trips
  FROM trips t JOIN zones z ON t.PULocationID=z.LocationID
  WHERE t.PULocationID IN (264,265)
  GROUP BY 1,2 ORDER BY trips DESC
""").df()

## 7. Temporal patterns

In [ ]:
hod = con.sql("""
  SELECT EXTRACT(HOUR FROM tpep_pickup_datetime) AS hour, COUNT(*) AS trips
  FROM trips GROUP BY 1 ORDER BY 1
""").df()
hod.plot(x='hour', y='trips', kind='bar', legend=False, figsize=(10,3))
plt.title('Trips by hour of day'); plt.tight_layout(); plt.show()

In [ ]:
con.sql("""
  SELECT dayname(tpep_pickup_datetime) AS dow, COUNT(*) AS trips,
         ROUND(AVG(fare_amount),2) AS avg_fare
  FROM trips WHERE fare_amount>0
  GROUP BY 1 ORDER BY trips DESC
""").df()

## 8. Tipping — the cash blind spot
Cash tips aren't recorded, so Cash trips show ~0% tips. Important caveat for any tip analysis.

In [ ]:
con.sql("""
  SELECT payment_type,
         COUNT(*) AS trips,
         ROUND(AVG(tip_amount),2) AS avg_tip,
         ROUND(100.0*AVG(CASE WHEN tip_amount>0 THEN 1 ELSE 0 END),1) AS pct_tipped
  FROM trips WHERE fare_amount>0
  GROUP BY 1 ORDER BY trips DESC
""").df()

## Takeaways → pipeline decisions
- **Driver-entered / unreliable fields**: `passenger_count` (nulls + 0s), `RatecodeID` (99/null). → filter `passenger_count BETWEEN 1 AND 6`.
- **Impossible values**: `fare_amount<=0`, `trip_distance<=0`, dropoff ≤ pickup. → DQ filters in `int_trips_enriched`.
- **Outliers**: multi-hour trips, huge fares. → cap duration to ≤180 min.
- **Cross-month leakage**: a few non-2023 rows. → year filter.
- **Undocumented codes**: `payment_type=0`, `RatecodeID=99`. → keep but be aware; don't assume the dictionary is exhaustive.
- **Cash tips invisible**: never read tip rates without segmenting by `payment_type`.
- **Geography is a join**: borough/zone only via the lookup; IDs 264/265 = unknown / outside NYC.

These are exactly the rules encoded in the dbt models — now you know *why* each exists.